# 03 - Mapping: one ANN vs 768 independent linear regressions (Stage 3)

For each pooling (`mean`, `cls`, `last`):

| arm | what it is |
|---|---|
| `identity` | no mapping at all - the control |
| `ridge` | 768 independent linear regressions, solved as one matrix |
| `mlp` | one 768->1024->768 network, early-stopped |

Evaluation: closed-set retrieval over the 2,000 test pairs, plus MSE / MAE / cosine.
`chance@k = k/2000` is reported too, so a small R@1 is readable as noise.


In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

from src import config as C, data, db, mapping, pipeline
print(C.ROOT)

In [ ]:
df = pipeline.stage_mapping()

In [ ]:
cols = ["mse", "mae", "cosine", "r@1", "r@5", "r@10", "rank_median"]
df[["pooling", "method", "hp"] + cols].round(4)

In [ ]:
df.pivot(index="pooling", columns="method", values="r@1").reindex(list(C.POOLINGS)).round(4)

In [ ]:
from IPython.display import Image, display

display(Image(filename=str(C.RESULTS / "retrieval_at1.png")))
display(Image(filename=str(C.RESULTS / "mlp_training_curve.png")))

**How to read this.**

* `mlp` ~ `ridge` ~ `identity`: the spaces are already close, or both are useless;
  a learned nonlinear map buys nothing.
* `mlp` much better than `ridge`: the relation is genuinely nonlinear - the
  interesting case, because a dynamic input-dependent map has something to model.
* `cls` / `last` behaving like `mean`: pooling does not matter, which weakens any
  "for each representation" claim.
